# Analyse du Data Drift

Le but est de comparer les données d’entraînement avec les données de production en utilisant Evidently.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from evidently import Report
from evidently.presets import DataDriftPreset

## Chargement des données

Les données de référence viennent du fichier CSV et les données de production sont stockées dans PostgreSQL.

In [2]:
if os.path.exists("/.dockerenv"):
    load_dotenv("docker.env")
else:
    load_dotenv("local.env")

df_train = pd.read_csv("df_total_clean.csv")
engine = create_engine(os.getenv("URLBDD"))
df_production = pd.read_sql('SELECT * FROM "predictions_inputs"', con=engine)

df_train = df_train.drop(columns=["SK_ID_CURR"])
df_production = df_production.drop(columns=["SK_ID_CURR"])

## Simulation du drift

Je garde uniquement les 25 % de clients avec les revenus les plus élevés en production pour simuler une différence avec les données d’entraînement.

In [3]:
seuil = df_production["AMT_INCOME_TOTAL"].quantile(0.75)
clients_riches = df_production[df_production["AMT_INCOME_TOTAL"] > seuil]
df_train_sample = df_train.sample(n=20000, random_state=42)

print("Clients sélectionnés :", len(clients_riches))

Clients sélectionnés : 19


## Rapport Evidently

Evidently compare les distributions des variables entre les deux jeux de données.

In [4]:
report = Report(metrics=[DataDriftPreset()])
resultat = report.run(reference_data=df_train_sample, current_data=clients_riches)
resultat.save_html("drift_report.html")

c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning:

invalid value encountered in divide

c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning:

invalid value encountered in divide



## Résultats : que signifie le 65 / 117 ?

Le rapport généré dans cette exécution indique **65 variables en drift sur 117**, soit **55,56 %**. C'est un signal important de différence entre **le jeu de référence** (échantillon aléatoire de 20 000 lignes d'entraînement) et **le jeu courant réellement analysé** (19 lignes, sélectionnées parmi les plus hauts revenus de la table `predictions_inputs`).

**À ne pas confondre :**
- 55,56 % est une **proportion de colonnes signalées**, **pas** « 55,56 % des clients ont dérivé » ni « le modèle a perdu 55,56 % de performance » ;
- l'analyse ne compare **pas toute la production** à l'entraînement : elle compare volontairement un **sous-groupe** de production au jeu d'entraînement ;
- le résultat décrit **cette exécution du rapport**, avec ces données et ces paramètres, et non un taux de drift garanti dans le temps.

Il faut consulter les **graphiques et tests de chaque variable** du fichier `drift_report.html` pour distinguer les changements de distribution réellement marqués, les différences modestes et les résultats instables. Le ratio global ne suffit pas à identifier les caractéristiques responsables.

## Interprétation : le protocole crée lui-même une différence

Le code calcule le 75e percentile du revenu `AMT_INCOME_TOTAL` en production, puis ne retient que les clients **strictement au-dessus** de ce seuil. Cela constitue une **simulation contrôlée de dérive** ou, plus précisément, une analyse d'un **sous-groupe sélectionné**.

1. **Revenu : effet attendu par construction.** La distribution de `AMT_INCOME_TOTAL` est tronquée vers les hauts revenus dans le groupe courant, contrairement à l'échantillon d'entraînement tiré aléatoirement. Une différence sur cette colonne est donc attendue ; ce n'est pas une découverte spontanée d'un changement du marché.
2. **Autres variables : possibles effets indirects.** Si les revenus sont associés au crédit demandé, aux annuités, au type d'emploi, à l'âge ou à d'autres caractéristiques du jeu, sélectionner les hauts revenus peut aussi modifier leurs distributions. C'est une **hypothèse de corrélation / biais de sélection**, pas la preuve que ces colonnes ont effectivement dérivé : il faut vérifier leurs résultats individuels dans le HTML.
3. **Lecture multivariée.** Plusieurs variables peuvent être liées entre elles. Ainsi, 65 alertes ne représentent pas nécessairement 65 phénomènes indépendants ; une même sélection de population peut produire de nombreuses différences associées.
4. **Pas de causalité démontrée.** Même si une variable varie avec le revenu, cela ne prouve pas que le revenu cause ce changement. Des facteurs communs ou la composition du groupe peuvent l'expliquer.

**Hypothèse principale :** une part substantielle des alertes peut être expliquée par le sous-échantillonnage volontaire des clients à revenus élevés et par les caractéristiques qui leur sont associées. Elle reste **à tester variable par variable**, et ne doit pas être présentée comme un résultat démontré par le seul décompte 65 / 117.

## Solidité statistique et limites

- **Seulement 19 clients dans le groupe courant** (nombre affiché par cette exécution) contre **20 000 dans la référence** : ce très fort déséquilibre peut rendre les distributions courantes instables et sensibles à quelques observations, notamment pour les variables catégorielles rares ou avec des valeurs extrêmes.
- **Sélection non représentative :** le groupe courant est construit à partir des revenus et ne représente donc pas l'ensemble des prédictions enregistrées dans `predictions_inputs`.
- **Dépendance aux tests et aux seuils :** le nombre de colonnes signalées dépend des méthodes de détection et des paramètres utilisés par Evidently ; le rapport doit être consulté pour connaître les décisions par colonne.
- **Multiplicité des comparaisons :** 117 variables sont examinées ; il faut éviter d'interpréter chaque signal isolément comme un changement métier certain.
- **Avertissements NumPy :** l'exécution a émis des avertissements de calcul liés à des divisions invalides. Certaines colonnes peuvent avoir une variance nulle ou provoquer des statistiques non définies ; **la cause exacte n'est pas établie** et mérite une vérification des colonnes concernées.
- **Drift des entrées ≠ dégradation prédictive :** le rapport ne mesure ici ni l'AUC, ni le rappel, ni les erreurs réelles du modèle. On ne peut pas conclure à une baisse de performance sans données de résultats fiables et mesures dédiées.
- **Aucun effet temporel isolé :** la sélection par revenu n'établit pas que les distributions ont évolué dans le temps.

Ces limites n'annulent pas l'utilité de la démonstration : elles précisent **ce qu'elle montre réellement**.

## Ce qu'il faut examiner dans le rapport HTML

Pour rendre l'analyse exploitable, ouvrir `drift_report.html` et relever pour les variables importantes :

| Vérification | Pourquoi c'est intéressant |
|---|---|
| **`AMT_INCOME_TOTAL`** : histogramme, quantiles, taux de drift | Vérifier l'effet voulu de la sélection des plus hauts revenus |
| **Montant du crédit / annuité**, si présents dans le rapport | Tester l'hypothèse d'un profil financier différent chez les clients aisés |
| **Âge, ancienneté professionnelle, catégories d'emploi**, si présents | Explorer une différence de composition de la clientèle plutôt qu'une évolution temporelle |
| **Variables non signalées en drift** | Identifier ce qui reste comparable malgré la sélection |
| **Colonnes presque constantes, rares ou avec valeurs manquantes** | Évaluer si certains résultats reposent sur peu d'observations |
| **Amplitude de variation, distribution et test statistique par variable** | Prioriser les écarts importants plutôt que compter uniquement les alertes |

**Important :** ces familles de variables sont des **pistes de lecture**, pas une liste vérifiée des 65 colonnes en drift. Ce notebook ne contient pas le tableau détaillé variable par variable ; ces constats doivent être établis en lisant le rapport généré.

## Hypothèses à tester et améliorations

**H1 — Effet de sélection :** une analyse de **toute** la table `predictions_inputs`, sans filtre sur les revenus, devrait présenter un profil de drift différent de celui des 19 hauts revenus. *Test :* générer un second rapport référence vs production complète avec les mêmes colonnes et une taille d'échantillon appropriée.

**H2 — Effet de taille d'échantillon :** une partie des alertes peut varier avec la faible taille du sous-groupe. *Test :* répéter l'expérience sur plusieurs sous-échantillons, étudier la stabilité des résultats et examiner les colonnes catégorielles peu représentées.

**H3 — Variables corrélées :** certains écarts autres que le revenu peuvent refléter la composition du groupe riche. *Test :* examiner les distributions, corrélations pertinentes et segments comparables, sans transformer une association en causalité.

**H4 — Risque pour le modèle :** si les nouveaux profils sont éloignés des profils d'entraînement, les prédictions pourraient être moins fiables pour ce segment. *Test :* comparer les probabilités produites, puis, dès que des étiquettes fiables sont disponibles, évaluer les performances par segment. Le drift seul ne démontre pas ce risque réalisé.

**Plan de monitoring réaliste :**
1. Établir un **rapport de référence** sur la production complète, sans intervention artificielle ;
2. surveiller périodiquement les variables importantes et l'évolution du volume de requêtes ;
3. documenter les seuils, examiner les faux signaux et déclencher une revue en cas d'alerte persistante ;
4. mesurer les performances métier dès que les résultats réels sont disponibles ;
5. n'envisager un réentraînement **qu'après diagnostic**, et non automatiquement parce qu'un rapport indique du drift.

## Conclusion utilisable en soutenance

> « J'ai simulé une dérive en isolant volontairement les clients de production appartenant au haut de la distribution des revenus. Evidently signale 65 variables sur 117, soit 55,56 %, dans cette comparaison précise. Le revenu est modifié par construction et d'autres différences peuvent venir de variables associées à ce segment. Le groupe ne contient toutefois que 19 observations, donc je ne peux pas assimiler ce résultat à un drift naturel de toute la production, ni à une perte mesurée de performance. Cette expérience démontre la capacité de mon dispositif à mettre en évidence des différences de distributions ; l'étape suivante serait de surveiller la production complète et de vérifier la stabilité et l'impact métier des alertes. »